# 🥉 Bronze — Ingestão

**Camada bronze:** os dados chegam praticamente como saíram da origem. O objetivo aqui é só **converter para um formato analítico decente** (Parquet) e tirar uma fotografia datada. Sem julgamentos, sem limpeza de negócio.

Vamos começar olhando o que temos na pasta e ir descobrindo cada coisa por vez.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path.cwd().parents[0] / 'data'
BRONZE = DATA / 'bronze'

list(BRONZE.iterdir())

[PosixPath('/home/ivissonalves/Workspaces/previsia/previsia-api/notebooks/data/bronze/cobranca_assessorias.csv'),
 PosixPath('/home/ivissonalves/Workspaces/previsia/previsia-api/notebooks/data/bronze/fluxo_pagamentos.xlsx')]

Temos dois arquivos brutos. Vamos abrir um por vez e ver o que está lá dentro.

## 1. `cobranca_assessorias.csv`

In [2]:
cobranca = pd.read_csv(BRONZE / 'cobranca_assessorias.csv')
cobranca.shape

(10000, 8)

In [3]:
cobranca.head()

,ID_Contrato,Nome_Assessoria,Data_Envio_Assessoria,Dias_Em_Atraso_Inicial,Valor_Inadimplente_Inicial,Status_Cobranca,Score_Interno_Risco,Regiao_Cliente
0,CONTR_2026_00001,Vértice Asset & Cobrança,2026-02-24,49,"R$ 25.007,89",Acordo Firmado,39.0,Sudeste
1,CONTR_2026_00002,Vértice Asset & Cobrança,2026-04-06,38,"R$ 8.448,59",Acordo Firmado,3.0,Sudeste
2,CONTR_2026_00003,Acerta Crédito Integrado,2026-02-05,152,"R$ 1.951,40",Insucesso,NaN,Sudeste
3,CONTR_2026_00004,Nexus Mediação Financeira,2026-04-24,84,4295.12,Em Aberto,68.0,Centro-Oeste
4,CONTR_2026_00005,Fênix Recuperação de Crédito,2026-02-06,52,4700.91,Acordo Firmado,30.0,Sudeste


10.000 linhas, 8 colunas. As colunas falam por si: um identificador de contrato, qual assessoria está cobrando, quando foi enviado, dias em atraso, valor, status, score interno e região do cliente.

Vamos olhar os tipos para entender como o pandas inferiu:

In [4]:
cobranca.dtypes

ID_Contrato                    object
Nome_Assessoria                object
Data_Envio_Assessoria          object
Dias_Em_Atraso_Inicial          int64
Valor_Inadimplente_Inicial     object
Status_Cobranca                object
Score_Interno_Risco           float64
Regiao_Cliente                 object
dtype: object

`Valor_Inadimplente_Inicial` veio como `object` — isso já dá um sinal de que tem texto misturado com número. Mas isso é assunto da **silver**. Aqui só vamos garantir que a data vire `datetime` antes de salvar (parquet preserva tipos):

In [5]:
cobranca['Data_Envio_Assessoria'] = pd.to_datetime(cobranca['Data_Envio_Assessoria'])
cobranca.dtypes

ID_Contrato                           object
Nome_Assessoria                       object
Data_Envio_Assessoria         datetime64[ns]
Dias_Em_Atraso_Inicial                 int64
Valor_Inadimplente_Inicial            object
Status_Cobranca                       object
Score_Interno_Risco                  float64
Regiao_Cliente                        object
dtype: object

Pronto. Vamos olhar uma amostra final só pra confirmar que nada se perdeu:

In [6]:
cobranca.sample(5, random_state=42)

,ID_Contrato,Nome_Assessoria,Data_Envio_Assessoria,Dias_Em_Atraso_Inicial,Valor_Inadimplente_Inicial,Status_Cobranca,Score_Interno_Risco,Regiao_Cliente
6252,CONTR_2026_06253,Vértice Asset & Cobrança,2026-02-27,99,"R$ 5.139,03",Acordo Firmado,29.0,Sudeste
4684,CONTR_2026_04685,Acerta Crédito Integrado,2026-01-30,88,4154.15,Em Aberto,11.0,Nordeste
1731,CONTR_2026_01732,Nexus Mediação Financeira,2026-01-15,157,"R$ 8.898,29",Ajuizado,34.0,Nordeste
4742,CONTR_2026_04743,Fênix Recuperação de Crédito,2026-03-21,108,"R$ 5.993,88",Em Aberto,11.0,Sudeste
4521,CONTR_2026_04522,Nexus Mediação Financeira,2026-02-03,55,"R$ 3.185,64",Em Aberto,99.0,Sudeste


## 2. `fluxo_pagamentos.xlsx`

In [7]:
fluxo = pd.read_excel(BRONZE / 'fluxo_pagamentos.xlsx', engine='openpyxl')
fluxo.shape

(100000, 9)

In [8]:
fluxo.head()

,ID_Pagamento,ID_Contrato,Numero_Parcela,Data_Vencimento,Data_Pagamento,Valor_Parcela,Valor_Pago,Forma_Pagamento,Indicador_Contemplado
0,PAG_0000001,CONTR_2026_07271,53,2025-06-27,2025-10-11,850,896.73,Boleto,Não
1,PAG_0000002,CONTR_2026_00861,12,2025-07-02,2025-06-29,850,850.00,Boleto,Sim
2,PAG_0000003,CONTR_2026_05391,54,2026-01-24,2026-01-19,450,450.00,Boleto,Não
3,PAG_0000004,CONTR_2026_05192,42,2025-12-03,2026-01-07,1500,1547.32,Boleto,Não
4,PAG_0000005,CONTR_2026_05735,6,2026-01-19,2026-01-14,1500,1500.00,Pix,Não


In [9]:
fluxo.dtypes

ID_Pagamento              object
ID_Contrato               object
Numero_Parcela             int64
Data_Vencimento           object
Data_Pagamento            object
Valor_Parcela              int64
Valor_Pago               float64
Forma_Pagamento           object
Indicador_Contemplado     object
dtype: object

100.000 parcelas. As datas o Excel já entregou como `datetime` — bom. `Indicador_Contemplado` veio como `object`: vamos espiar:

In [10]:
fluxo['Indicador_Contemplado'].value_counts(dropna=False)

Indicador_Contemplado
Não    69986
Sim    30014
Name: count, dtype: int64

Texto `Sim`/`Não`. Conversão para booleano é assunto da silver. Bronze mantém como veio.

## 3. Persistência bronze

Salvamos como parquet usando o mesmo nome lógico. Da silver em diante, ninguém precisa mais saber o formato original.

In [11]:
cobranca.to_parquet(BRONZE / 'cobranca_assessorias.parquet', index=False)
fluxo.to_parquet(BRONZE / 'fluxo_pagamentos.parquet', index=False)

for f in BRONZE.glob('*.parquet'):
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name:35s} {size_mb:6.2f} MB')

  cobranca_assessorias.parquet          0.20 MB
  fluxo_pagamentos.parquet              1.30 MB


Bronze concluído. Os dois arquivos brutos viraram parquet (mais compactos e tipados) sem qualquer alteração de conteúdo. Próximo passo: **silver** — descobrir e corrigir os problemas de qualidade.